<a href="https://colab.research.google.com/github/ishoyy/MovieLens/blob/main/Movie_Lens_Matrix_Factorization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **Q1** (20 marks) **Matrix Factorization for Collaborative Filtering**

We investigate matrix factorization for collaborative filtering using [the MovieLens 100K dataset]( https://grouplens.org/datasets/movielens/100k/), which contains 100,000 ratings provided by 943 users on 1,682 movies.
In this question, you will first follow the given code to download the MovieLens 100K ratings and load them into a Python sparse matrix. After that, refer to [this matrix factorization implementation](https://colab.research.google.com/drive/1WVEKcaHB858WsdmjQ1MOjyH2G92TDveF) to factorize the user–movie rating matrix for recommendation purposes (do NOT use `jax.grad` here).
Once the factorization is complete, use the cosine similarity between movie vectors, defined as

$$
\text{sim}(\mathbf{v}_i,\mathbf{v}_j) = \frac{\mathbf{v}_i \cdot \mathbf{v}_j}{\Vert \mathbf{v}_i \Vert \Vert \mathbf{v}_j \Vert}
$$
to measure the similarity between different movies.

As an example, you may recommend the top three most similar movies for each of the following movie IDs:

$$
\Big\{0, 94, 250, 379, 784, 863, 1044, 1141, 1203, 1445 \Big\}
$$

You may refer to [the movie list](http://www.cs.yorku.ca/~huijiang/ml-100k/u.item) for the corresponding movie titles.
(Note: the indexes in the list start from 1, not 0.)

*   Use both Stochastic Gradient Descent (SGD) and the Alternating algorithm for matrix factorization. Compare their performance in terms of training speed and recommendation quality.
* Evaluate how the movie vector dimension size $k=10, 30, 100$
affects both the training efficiency and the quality of recommendations.






In [ ]:
## Preparation code to download the data and load them into a sparse matrix
##

import pandas as pd
import numpy as np
import gdown
from scipy.sparse import csr_matrix
import time
from numpy.linalg import solve
from sklearn.metrics.pairwise import cosine_similarity


# 1. Download and unzip the dataset first
#    URL: https://files.grouplens.org/datasets/movielens/ml-100k.zip
#    Then point to the u.data file inside the unzipped folder.
#!wget -O ml-100k.zip https://files.grouplens.org/datasets/movielens/ml-100k.zip
#!unzip -o -q ml-100k.zip
#path = "ml-100k/u.data"

##download from google drive
#url = 'https://drive.google.com/file/d/12cpRJ2977nyqUSiUMkvdRS9OtYy6Fj_P/view?usp=share_link'
#path = 'u.data'
#gdown.download(url, path, quiet=False,fuzzy=True)

# download from YorkU server
!wget -q -r -np -nH --cut-dirs=1 -R "index.html*" http://www.cs.yorku.ca/~huijiang/ml-100k/
path = "ml-100k/u.data"

# The raw file is tab-separated: user_id, item_id, rating, timestamp

df = pd.read_csv(path, sep='\t', names=['user_id', 'item_id', 'rating', 'timestamp'])
print(df);

# 2. Convert IDs to 0-based indices (good for matrix indexing)
df['user_id'] = df['user_id'].astype('category')
df['item_id'] = df['item_id'].astype('category')


# extracting the unique users and movies so we dont have any missing data in. our training set
n_users = df['user_id'].nunique()
n_items = df['item_id'].nunique()

print(f"Users: {n_users}, Items: {n_items}, Ratings: {len(df)}")

# 3. Build a sparse user–item rating matrix (CSR format)
# Rows → users, Columns → items, Values → ratings
X = csr_matrix(
    (df['rating'].astype(float),
     (df['user_id'].cat.codes, df['item_id'].cat.codes)),
    shape=(n_users, n_items)
)

print(X.shape)

# A sparse matrix is a type of matrix in which a majority of its elements are zero, memory requirement reduction made by storing only non-zero entries
# CSR format: 3 one-dimensional arrays, V = nonzero values, COL_INDEX = column indexes, ROW_INDEX = row indexes. then, extract rows to compress row-indexes
# ex: rating 5 given by user_id = 0, item_id = 5 (0, 0)

# for visualization purposes
X_sparse_matrix = X.toarray();
print (X_sparse_matrix);

# matrix factorization:
# let X be the ratings matrix (target matrix), P is the item matrix (1682 x k), Q is the user matrix (943 x k). k is the number of features (hyperparameters) for each user id/item id
# take dot product of P and Q, compare to values in target matrix X, adjust values until difference between predicted value and actual value is minimized (schotastic gradient descent)
# For the above, schotastic gradient descent will work by looping through each rating in the training data, try to predict the rating, aand calculate a prediction error then updating the data until
# convergence is found
# we can maybe use the Root Mean Squared Error that represents the standard deviation between a set of estimated values to the actual values of the ratings
# RMSE = sqrt(1/n summation(value1 - value2)^2)


k_values = [10, 30, 50, 100]
eta = 0.01 #step size
lam = 0.1 # regularization
epochs = 10 # number of passes



# we need to extract all non-zero entries from the ratings matrix in X (sparse matrix)
rows, cols = X.nonzero()
ratings = X.data      # actual rating values
results = [];
# Trainig data

start_time = time.time(); #to help evaluate training efficiency

for k in k_values:
    print(f"\n--- Training with k = {k} ----")
    # initialize random values for P (|user_id| x k) and Q (|item_id| x k) before SGD
    P = 0.1 * np.random.randn(n_users, k)
    Q = 0.1 * np.random.randn(n_items, k)

    for epoch in range(epochs):
        total_err = 0.0

        for u, i, r in zip(rows, cols, ratings):

            pred = P[u, :].dot(Q[i, :].T)   # prediction is the rating. P[u, :] returns a 1xk vector of user u's features, Q[i, :] for 1xk vector of item features

            err = r - pred
            total_err += err * err     # how wrong is the model for all ratings in this epoch?

            #update SGD
            P[u, :] += eta * (err * Q[i, :] - lam * P[u, :])
            Q[i, :] += eta * (err  * P[u, :] - lam * Q[i, :])

        rmse = np.sqrt(total_err/ len(ratings))   # average size of prediction mistakes across all ratings
        print(f"Epoch {epoch+1}/{epochs}, RMSE = {rmse:.4f}") #good enough ?

    elapsed_time = time.time() - start_time
    print(f"Training time: {elapsed_time:.2f} seconds")
    results.append((k, rmse, elapsed_time))


# Print summary
print("\nSummary of results:")
print("k\tFinal RMSE\tTraining Time (s)")
for k, rmse, t in results:
    print(f"{k}\t{rmse:.4f}\t\t{t:.2f}")


# ------------------------------------------------------------------
# Matrix factorization using Alternating algorithm


# X csr matrix remains the same
# hyperparameters
k = 20
lam = 0.1
epochs = 10

# initializing latent matrixes
P = np.random.randn(n_users, k) * 0.1  # user features
Q = np.random.randn(n_items, k) * 0.1  # item/movie features

# ALS training loop
for epoch in range(epochs):
    #updating p
    for u in range(n_users):
        item_idx = X[u, :].nonzero()[1]  # items rated by user u
        Q_u = Q[item_idx, :]             # movie features for rated items
        r_u = X[u, item_idx].toarray().flatten()  # ratings by user u
        if len(item_idx) > 0:  # we want to avoid  empty slices
            A = Q_u.T @ Q_u + lam * np.eye(k)
            b = Q_u.T @ r_u
            P[u, :] = solve(A, b)

    # updating q
    for i in range(n_items):
        user_idx = X[:, i].nonzero()[0]  # users who rated item i
        P_i = P[user_idx, :]             # user features for those ratings
        r_i = X[user_idx, i].toarray().flatten()  # ratings for item i
        if len(user_idx) > 0:
            A = P_i.T @ P_i + lam * np.eye(k)
            b = P_i.T @ r_i
            Q[i, :] = solve(A, b)

    # Optional: compute RMSE
    pred = P @ Q.T
    mask = X.toarray() != 0
    rmse = np.sqrt(np.sum((mask * (X.toarray() - pred))**2) / mask.sum())
    print(f"Epoch {epoch+1}/{epochs}, RMSE = {rmse:.4f}")

# computing cosine similarity between movies where each row in Q is a movie vector
movie_sim = cosine_similarity(Q)  # shape: (n_items x n_items)

# ex // find top 5 similar movies to movie_id = 0
movie_id = 0
similar_movies_idx = np.argsort(-movie_sim[movie_id, :])[1:6]  # exclude self
print(f"Top 5 movies similar to movie {movie_id}: {similar_movies_idx}")
print(f"Similarity scores: {movie_sim[movie_id, similar_movies_idx]}")




       user_id  item_id  rating  timestamp
0          196      242       3  881250949
1          186      302       3  891717742
2           22      377       1  878887116
3          244       51       2  880606923
4          166      346       1  886397596
...        ...      ...     ...        ...
99995      880      476       3  880175444
99996      716      204       5  879795543
99997      276     1090       1  874795795
99998       13      225       2  882399156
99999       12      203       3  879959583

[100000 rows x 4 columns]
Users: 943, Items: 1682, Ratings: 100000
(943, 1682)
[[5. 3. 4. ... 0. 0. 0.]
 [4. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [5. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 5. 0. ... 0. 0. 0.]]

--- Training with k = 10 ----
Epoch 1/10, RMSE = 2.6494
Epoch 2/10, RMSE = 1.2695
Epoch 3/10, RMSE = 1.0212
Epoch 4/10, RMSE = 0.9795
Epoch 5/10, RMSE = 0.9640
Epoch 6/10, RMSE = 0.9556
Epoch 7/10, RMSE = 0.9500
Epoch 8/10, RMSE = 0.9456
Epoch 9/1